In [19]:
pip install music21 tensorflow numpy


Note: you may need to restart the kernel to use updated packages.


In [25]:
# ==== STEP 1: IMPORTS ====
import glob
import pickle
import numpy as np
from music21 import converter, instrument, note, chord, stream
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense, Activation

# ==== STEP 2: LOAD NOTES FROM MIDI FILES ====
def get_notes():
    notes = []
    midi_files = glob.glob("midi_songs/*.mid")

    if len(midi_files) == 0:
        raise ValueError("❌ No MIDI files found in 'midi_songs/' folder. Add .mid files to continue.")

    for file in midi_files:
        print(f"📥 Loading {file}")
        try:
            midi = converter.parse(file)
            parts = instrument.partitionByInstrument(midi)
            elements = parts.parts[0].recurse() if parts else midi.flat.notes
            for element in elements:
                if isinstance(element, note.Note):
                    notes.append(str(element.pitch))
                elif isinstance(element, chord.Chord):
                    notes.append('.'.join(str(n) for n in element.normalOrder))
        except Exception as e:
            print(f"⚠️ Skipping {file} due to error: {e}")

    if not notes:
        raise ValueError("❌ No notes found. Check your MIDI files.")

    # Save for reuse
    with open("notes.pkl", "wb") as f:
        pickle.dump(notes, f)

    print(f"✅ Total notes collected: {len(notes)}")
    return notes

# ==== STEP 3: PREPARE SEQUENCES FOR TRAINING ====
def prepare_sequences(notes, sequence_length=30):
    if len(notes) <= sequence_length:
        raise ValueError("❌ Not enough notes to generate sequences. Reduce sequence length or add more MIDI files.")

    pitchnames = sorted(set(notes))
    note_to_int = {note: number for number, note in enumerate(pitchnames)}

    network_input = []
    network_output = []

    for i in range(len(notes) - sequence_length):
        seq_in = notes[i:i + sequence_length]
        seq_out = notes[i + sequence_length]
        network_input.append([note_to_int[n] for n in seq_in])
        network_output.append(note_to_int[seq_out])

    n_patterns = len(network_input)
    network_input = np.reshape(network_input, (n_patterns, sequence_length, 1))
    network_input = network_input / float(len(pitchnames))
    network_output = to_categorical(network_output)

    print(f"✅ Prepared {n_patterns} input sequences.")
    return network_input, network_output, note_to_int, pitchnames

# ==== STEP 4: BUILD LSTM MODEL ====
def create_network(network_input, output_size):
    model = Sequential()
    model.add(LSTM(256, input_shape=(network_input.shape[1], network_input.shape[2]), return_sequences=True))
    model.add(Dropout(0.3))
    model.add(LSTM(256))
    model.add(Dropout(0.3))
    model.add(Dense(256))
    model.add(Dropout(0.3))
    model.add(Dense(output_size))
    model.add(Activation('softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    print("✅ Model created.")
    return model

# ==== STEP 5: GENERATE MUSIC ====
def generate_notes(model, network_input, note_to_int, pitchnames, length=34):
    int_to_note = {number: note for note, number in note_to_int.items()}
    start = np.random.randint(0, len(network_input) - 1)
    pattern = network_input[start].tolist()
    prediction_output = []

    def sample(predictions, temperature=0.8):
        predictions = np.log(predictions + 1e-8) / temperature
        exp_preds = np.exp(predictions)
        predictions = exp_preds / np.sum(exp_preds)
        return np.random.choice(len(predictions), p=predictions)

    for note_index in range(length):
        input_seq = np.reshape(pattern, (1, len(pattern), 1))
        input_seq = input_seq / float(len(pitchnames))

        prediction = model.predict(input_seq, verbose=0)
        index = sample(prediction[0], temperature=0.8)  # use randomness
        result = int_to_note[index]
        prediction_output.append(result)

        pattern.append([index])
        pattern = pattern[1:]

    return prediction_output


# ==== STEP 6: SAVE OUTPUT TO MIDI ====
def create_midi(prediction_output, output_name="output.mid"):
    output_notes = []
    for pattern in prediction_output:
        if ('.' in pattern) or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes_list = [note.Note(int(n)) for n in notes_in_chord]
            new_chord = chord.Chord(notes_list)
            output_notes.append(new_chord)
        else:
            new_note = note.Note(pattern)
            output_notes.append(new_note)

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp=output_name)
    print(f"🎵 Music saved as: {output_name}")

# ==== STEP 7: RUN THE FULL PIPELINE ====
if __name__ == "__main__":
    print("🎼 Starting AI Music Generator...")

    try:
        notes = get_notes()
        net_in, net_out, note_to_int, pitchnames = prepare_sequences(notes)
        model = create_network(net_in, len(pitchnames))

        print("🧠 Training model (please wait)...")
        model.fit(net_in, net_out, epochs=50, batch_size=64)

        print("🎶 Generating music...")
        output = generate_notes(model, net_in, note_to_int, pitchnames)
        create_midi(output, output_name="output.mid")

        print("✅ Done! Play 'output.mid' to hear your AI-generated music.")
    except Exception as e:
        print(f"❌ ERROR: {e}")


🎼 Starting AI Music Generator...
📥 Loading midi_songs\a-3.mid
📥 Loading midi_songs\a-4.mid
📥 Loading midi_songs\a-5.mid
📥 Loading midi_songs\a3.mid
📥 Loading midi_songs\a4.mid
📥 Loading midi_songs\a5.mid
📥 Loading midi_songs\b3.mid
📥 Loading midi_songs\b4.mid
📥 Loading midi_songs\b5.mid
📥 Loading midi_songs\c-3.mid
📥 Loading midi_songs\c-4.mid
📥 Loading midi_songs\c-5.mid
📥 Loading midi_songs\c3.mid
📥 Loading midi_songs\c4.mid
📥 Loading midi_songs\c5.mid
📥 Loading midi_songs\c6.mid
📥 Loading midi_songs\d-3.mid
📥 Loading midi_songs\d-4.mid
📥 Loading midi_songs\d-5.mid
📥 Loading midi_songs\d3.mid
📥 Loading midi_songs\d4.mid
📥 Loading midi_songs\d5.mid
📥 Loading midi_songs\e3.mid
📥 Loading midi_songs\e4.mid
📥 Loading midi_songs\e5.mid
📥 Loading midi_songs\f-3.mid
📥 Loading midi_songs\f-4.mid
📥 Loading midi_songs\f-5.mid
📥 Loading midi_songs\f3.mid
📥 Loading midi_songs\f4.mid
📥 Loading midi_songs\f5.mid
📥 Loading midi_songs\g-3.mid
📥 Loading midi_songs\g-4.mid
📥 Loading midi_songs\g-5.mid


C:\Users\nadee\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


✅ Model created.
🧠 Training model (please wait)...
Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - loss: 3.5922
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - loss: 3.5000
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 3.3989
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 3.2114
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - loss: 2.7163
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - loss: 2.2704
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - loss: 2.2973
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - loss: 2.2346
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - loss: 2.3560
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - loss: 2.0175
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - loss: 2.1801
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - loss: 2.0281
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - loss: 2.2259
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - loss: 2.0953
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 